# 04 · Simple spectral extraction

So far we've gone *forward*: source spectrum → dispersed image. This notebook goes *backward* — pull a 1D spectrum back out of a dispersed image. `roman_disperser` has no extraction routine, so we build a minimal **boxcar** extractor from the optical model itself, in three steps: (1) the **trace** maps wavelength → detector pixel; (2) **autodiff** of the trace gives the dispersion dλ/pixel; (3) sum the cross-dispersion pixels at each wavelength and rescale. We check it against the known input spectrum, then package it for reuse.

This is a teaching extractor (boxcar, single source, no contamination handling) — enough to recover a clean spectrum and to drive the line-profile lesson in notebook 05.

## 0 · Setup — make a dispersed star to extract

In [ ]:
import os
from pathlib import Path
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(Path.home() / ".cache" / "roman_grs_jax"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import jax
import jax.numpy as jnp

from roman_disperser import paths, psf_model, star_disperser
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

SCA = 5
model = RomanOpticalModel(config_file=str(paths.optical_model_path()))
opt = omj.make_sca_payload(model, sca=SCA, order="1")
psf = psf_model.get_or_make_psf_payload(detector=f"WFI{SCA:02d}", order="1",
                                        cache_dir=str(paths.psf_cache_dir()), verbose=False)
star_disp = star_disperser.make_star_disperser(psf, opt)

wl_um, _, _ = th.grism_wavelength_grid()
X_STAR, Y_STAR = 2000.0, 2000.0
_, counts_in = th.template_to_counts("g0v", 16.0, sca=SCA, order="1", wl_um=wl_um)
img = np.asarray(star_disp(X_STAR, Y_STAR, jnp.asarray(wl_um),
                           jnp.asarray(counts_in), jnp.zeros((4088, 4088), jnp.float32)))
print(f"dispersed a G0V star (the 'observation' we'll extract); {wl_um.size} wavelengths")

## 1 · The spectral trace

The optical model tells us, for an undispersed source at `(x, y)`, where light of each wavelength lands. We chain the JAX transforms `sca_to_fpa → trace_beam → mpa_to_sca` to get the **trace**: pixel position as a function of wavelength. Overlaid on the dispersed image, it follows the spectrum exactly. (Dispersion runs along **y**.)

In [ ]:
def spectral_trace(payload, xsca, ysca, wl_um):
    wl = jnp.asarray(wl_um)
    xfpa, yfpa = omj.sca_to_fpa(payload, xsca, ysca)
    xmpa, ympa = omj.trace_beam(payload, jnp.broadcast_to(xfpa, wl.shape),
                                jnp.broadcast_to(yfpa, wl.shape), wl)
    tx, ty = omj.mpa_to_sca(payload, xmpa, ympa)
    return np.asarray(tx).ravel(), np.asarray(ty).ravel()

trace_x, trace_y = spectral_trace(opt, X_STAR, Y_STAR, wl_um)

ys, xs = np.nonzero(img)
pad = 15
bx0, bx1, by0, by1 = xs.min()-pad, xs.max()+pad, ys.min()-pad, ys.max()+pad
fig, ax = plt.subplots(figsize=(4, 7))
ax.imshow(img[by0:by1, bx0:bx1], origin="lower", cmap="inferno", extent=[bx0, bx1, by0, by1],
          norm=AsinhNorm(linear_width=img.max()*0.002, vmin=0, vmax=img.max()))
ax.plot(trace_x - 1, trace_y - 1, color="cyan", lw=0.8, label="optical-model trace")
ax.legend(loc="upper right", fontsize=8)
ax.set(title="dispersed star + trace", xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

## 2 · The dispersion, by autodiff

To turn "counts per pixel-row" into "counts per wavelength bin" we need the local dispersion **dy/dλ** (pixels per micron). Because the trace is a differentiable JAX function, we get it exactly with `jax.grad` — no finite differences — and `vmap` it over the wavelength grid.

In [ ]:
def trace_y_only(payload, xsca, ysca, wl):
    wl1 = jnp.atleast_1d(wl)
    xfpa, yfpa = omj.sca_to_fpa(payload, xsca, ysca)
    xmpa, ympa = omj.trace_beam(payload, jnp.broadcast_to(xfpa, wl1.shape),
                                jnp.broadcast_to(yfpa, wl1.shape), wl1)
    _, ty = omj.mpa_to_sca(payload, xmpa, ympa)
    return ty[0]

dydl_fn = jax.jit(jax.vmap(jax.grad(lambda w: trace_y_only(opt, X_STAR, Y_STAR, w))))
dy_dlam = np.asarray(dydl_fn(jnp.asarray(wl_um)))     # pixels per micron

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wl_um, np.abs(dy_dlam), color="C2")
ax.set(xlabel="wavelength [µm]", ylabel="|dy/dλ| [pix/µm]", title="dispersion along the trace")
fig.tight_layout()
print(f"~{np.abs(dy_dlam).mean():.0f} pix/µm  ≈  {1e4/np.abs(dy_dlam).mean():.1f} Å per pixel")

## 3 · Boxcar extraction

At each wavelength, sum the pixels within ±`aperture` of the trace in the cross-dispersion (x) direction, then multiply by `|dy/dλ|·Δλ` to convert to a count rate per wavelength bin. Compared against the spectrum we fed in, the recovery is excellent.

In [ ]:
dlam_um = wl_um[1] - wl_um[0]
aperture = 12
extracted = np.zeros(len(wl_um))
for i in range(len(wl_um)):
    ix = int(round(trace_x[i])) - 1       # 1-indexed FITS -> 0-indexed array
    iy = int(round(trace_y[i])) - 1
    if 0 <= ix < img.shape[1] and 0 <= iy < img.shape[0]:
        extracted[i] = img[iy, max(0, ix-aperture):ix+aperture+1].sum()
extracted = extracted * np.abs(dy_dlam) * dlam_um

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wl_um, counts_in, color="C0", lw=1.3, label="input spectrum")
ax.plot(wl_um, extracted, color="C3", lw=0.8, label=f"extracted (±{aperture} pix)")
ax.set(xlabel="wavelength [µm]", ylabel="count rate [e⁻/s per bin]",
       title="extracted vs input")
ax.legend(); fig.tight_layout()

m = (extracted > 0) & (counts_in > 0)
print(f"recovered {np.median(extracted[m]/counts_in[m])*100:.0f}% of the flux at ±{aperture} pix")

## 4 · Aperture, and what's left out

A boxcar trades completeness for simplicity: a wider aperture captures more of the PSF wings (more flux) at the cost of more background and contamination. The library has no extractor, so the function above is packaged as `tutorial_helpers.extract_1d` for the next notebook.

In [ ]:
for ap in [4, 8, 12, 20]:
    e = th.extract_1d(img, opt, X_STAR, Y_STAR, wl_um, aperture=ap)
    m = (e > 0) & (counts_in > 0)
    print(f"aperture ±{ap:2d} pix → {np.median(e[m]/counts_in[m])*100:.0f}% of the flux")

This extractor assumes the source is isolated. In a real field, a neighbour's **0th order** or an overlapping 1st-order trace can land in the aperture and masquerade as a feature — which is exactly why a *roll* (notebook 03) helps separate real lines from contaminants.

## Recap

- The **trace** (`sca_to_fpa → trace_beam → mpa_to_sca`) maps wavelength → pixel; **`jax.grad`** of it gives the dispersion.
- A **boxcar** sum along the cross-dispersion direction, scaled by `|dy/dλ|·Δλ`, recovers the 1D spectrum (~90%+ of the flux for a modest aperture).
- Packaged as `tutorial_helpers.extract_1d(image, payload, x, y, wl_um, aperture=...)`.

**Next — [05 · Position angle and emission-line profiles](05_pa_line_profiles.ipynb).** We disperse an *elongated* galaxy with an emission line and watch the extracted line width change with the galaxy's orientation — morphological broadening, a real systematic in slitless spectroscopy.